# Streaming Mock Transactions → Eventhouse

**FabricIQ-L300 — Lab support notebook**

This notebook generates synthetic Thai retail-banking streaming events and pushes them into a Fabric **Eventhouse** (KQL Database) so the labs have a live event stream to work with.

Two streams are produced:

| Stream | KQL table | Rate (default) |
|---|---|---|
| `card_transactions` | `CardTransactions` | 5 events/sec |
| `deposit_transactions` | `DepositTransactions` | 3 events/sec |

## Run modes

1. **Pure Python (driver-only)** — uses `azure-kusto-data` / `azure-kusto-ingest`. Good for steady, low-volume mock streams. Default mode in this notebook.
2. **Spark Structured Streaming** — uses the Fabric Spark Kusto connector. Use when you need higher throughput. Cells are provided at the bottom.

## Prerequisites

- A Fabric workspace with an **Eventhouse** created.
- A KQL database inside that Eventhouse (note the **Query URI** — looks like `https://<eventhouse>.<region>.kusto.fabric.microsoft.com`).
- The notebook runs as a user / service principal that has **Database Ingestor** or **Database Admin** role on the KQL DB.
- The two destination tables exist (the setup cell below creates them with `.create-merge table`).

## 1. Configuration — edit me

In [ ]:
# === EDIT THESE THREE VALUES ===
KUSTO_CLUSTER_URI = "https://<your-eventhouse>.<region>.kusto.fabric.microsoft.com"
KUSTO_DATABASE    = "FabricIQ_L300"
TENANT_ID         = "<your-tenant-guid>"  # only needed for service principal auth

# === Stream tuning ===
CARD_RATE_PER_SEC    = 5
DEPOSIT_RATE_PER_SEC = 3
RUN_DURATION_SECONDS = 600   # 10 minutes; set to None to run forever
BATCH_SIZE           = 50    # events per ingest call

CARD_TABLE    = "CardTransactions"
DEPOSIT_TABLE = "DepositTransactions"

## 2. Install dependencies (skip if already installed in the Fabric runtime)

In [ ]:
%pip install --quiet azure-kusto-data azure-kusto-ingest azure-identity

## 3. Authenticate

Inside Fabric notebooks the simplest path is `notebookutils.credentials.getToken('kusto')` to grab the workspace user's Kusto token. Outside Fabric, fall back to `DefaultAzureCredential`.

In [ ]:
from azure.kusto.data import KustoConnectionStringBuilder, KustoClient
from azure.kusto.ingest import QueuedIngestClient, IngestionProperties, DataFormat, StreamDescriptor

try:
    # In Fabric: pass-through current user identity
    import notebookutils  # type: ignore
    access_token = notebookutils.credentials.getToken('kusto')
    kcsb_query   = KustoConnectionStringBuilder.with_aad_user_token_authentication(KUSTO_CLUSTER_URI, access_token)
    kcsb_ingest  = KustoConnectionStringBuilder.with_aad_user_token_authentication(KUSTO_CLUSTER_URI, access_token)
    print('✓ Auth via Fabric notebookutils')
except Exception:
    # Outside Fabric: device code or DefaultAzureCredential
    kcsb_query   = KustoConnectionStringBuilder.with_az_cli_authentication(KUSTO_CLUSTER_URI)
    kcsb_ingest  = KustoConnectionStringBuilder.with_az_cli_authentication(KUSTO_CLUSTER_URI)
    print('✓ Auth via az CLI / DefaultAzureCredential')

kusto_client  = KustoClient(kcsb_query)
ingest_client = QueuedIngestClient(kcsb_ingest)

## 4. Create destination tables (idempotent)

In [ ]:
create_card_kql = f"""
.create-merge table {CARD_TABLE} (
    event_id: string,
    event_time: datetime,
    card_id: string,
    customer_id: string,
    card_type: string,
    channel: string,
    merchant_name: string,
    mcc_code: string,
    mcc_category: string,
    amount_thb: real,
    currency: string,
    auth_status: string,
    country: string,
    is_fraud_suspect: bool
)
"""

create_deposit_kql = f"""
.create-merge table {DEPOSIT_TABLE} (
    event_id: string,
    event_time: datetime,
    account_id: string,
    customer_id: string,
    txn_type: string,
    channel: string,
    amount_thb: real,
    currency: string,
    balance_after_thb: real,
    counter_account: string,
    branch_id: string
)
"""

# Enable streaming ingestion (faster than queued for small batches)
enable_streaming_kql = f"""
.alter table {CARD_TABLE} policy streamingingestion enable
.alter table {DEPOSIT_TABLE} policy streamingingestion enable
"""

for stmt in [create_card_kql, create_deposit_kql]:
    kusto_client.execute_mgmt(KUSTO_DATABASE, stmt)
print('✓ Tables created / merged')

for stmt in enable_streaming_kql.strip().split('\n'):
    if stmt.strip():
        try:
            kusto_client.execute_mgmt(KUSTO_DATABASE, stmt)
        except Exception as e:
            print(f'(streaming policy note) {e}')
print('✓ Streaming ingestion policy applied')

## 5. Mock event generator (Thai retail-banking flavor)

In [ ]:
import random, uuid
from datetime import datetime, timezone

THAI_MERCHANTS = [
    '7-Eleven Sukhumvit',  'Tops Daily Asok',   'Big C Rajdamri',
    'CP All HQ',           "Lotus's Phra Ram 4",'Central Embassy',
    'Siam Paragon',        'ICONSIAM',          'Terminal 21 Asok',
    'MBK Center',          'Lazada TH',         'Shopee TH',
    'Grab Food',           'LINE MAN',          'AIS Online',
    'True Online',         'BTS SkyTrain',      'MRT Bangkok',
    'PTT Station',         'Bangchak Petrol',   'Starbucks Silom',
    'Café Amazon',         'After You Siam',    'Krispy Kreme',
]
MCC_CODES = [
    ('5411','Grocery'), ('5812','Restaurant'), ('5541','Fuel'),
    ('4111','Transport'),('5732','Electronics'),('5651','Apparel'),
    ('5999','Misc Retail'),('4814','Telecom'),
]
DEPOSIT_TYPES = ['DEPOSIT','WITHDRAW','TRANSFER_IN','TRANSFER_OUT','PROMPTPAY_IN','PROMPTPAY_OUT','FEE']

def make_card_event() -> dict:
    mcc = random.choice(MCC_CODES)
    return {
        'event_id':      str(uuid.uuid4()),
        'event_time':    datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'card_id':       f'CRD{random.randint(1,7500):07d}',
        'customer_id':   f'CUS{random.randint(1,10000):07d}',
        'card_type':     random.choice(['Credit','Debit']),
        'channel':       random.choice(['POS','Ecommerce','Contactless','ATM','QR']),
        'merchant_name': random.choice(THAI_MERCHANTS),
        'mcc_code':      mcc[0],
        'mcc_category':  mcc[1],
        'amount_thb':    round(random.uniform(50, 50000), 2),
        'currency':      'THB',
        'auth_status':   random.choices(['APPROVED','DECLINED'], weights=[0.95,0.05])[0],
        'country':       'TH',
        'is_fraud_suspect': random.random() < 0.01,
    }

def make_deposit_event() -> dict:
    t = random.choice(DEPOSIT_TYPES)
    return {
        'event_id':     str(uuid.uuid4()),
        'event_time':   datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'account_id':   f'ACC{random.randint(1,18000):08d}',
        'customer_id':  f'CUS{random.randint(1,10000):07d}',
        'txn_type':     t,
        'channel':      random.choice(['Mobile App','ATM','Branch','Internet Banking','PromptPay']),
        'amount_thb':   round(random.uniform(100, 200000), 2),
        'currency':     'THB',
        'balance_after_thb': round(random.uniform(0, 5_000_000), 2),
        'counter_account': f'{random.randint(100,999)}-{random.randint(0,9)}-{random.randint(10000,99999)}-{random.randint(0,9)}'
                            if 'TRANSFER' in t or 'PROMPTPAY' in t else None,
        'branch_id':    f'BR{random.randint(1,50):04d}' if random.random() < 0.3 else None,
    }

print('Sample card:   ', make_card_event())
print('Sample deposit:', make_deposit_event())

## 6. Ingest helper (streaming JSON → Kusto)

In [ ]:
import io, json

def ingest_batch(table: str, events: list[dict]) -> None:
    if not events:
        return
    payload = '\n'.join(json.dumps(e, ensure_ascii=False) for e in events).encode('utf-8')
    stream  = io.BytesIO(payload)
    props   = IngestionProperties(
        database=KUSTO_DATABASE,
        table=table,
        data_format=DataFormat.MULTIJSON,
    )
    ingest_client.ingest_from_stream(StreamDescriptor(stream), ingestion_properties=props)
    print(f'  → {table}: {len(events)} events queued')

## 7. Run the producer

Generates batches at the configured rate and ingests them. Stops after `RUN_DURATION_SECONDS`. Interrupt the cell to stop early.

In [ ]:
import time

start = time.time()
card_buf, dep_buf = [], []
card_total = dep_total = 0

try:
    while True:
        elapsed = time.time() - start
        if RUN_DURATION_SECONDS is not None and elapsed >= RUN_DURATION_SECONDS:
            break
        # one second worth of events
        for _ in range(CARD_RATE_PER_SEC):    card_buf.append(make_card_event())
        for _ in range(DEPOSIT_RATE_PER_SEC): dep_buf.append(make_deposit_event())

        if len(card_buf) >= BATCH_SIZE:
            ingest_batch(CARD_TABLE, card_buf); card_total += len(card_buf); card_buf = []
        if len(dep_buf) >= BATCH_SIZE:
            ingest_batch(DEPOSIT_TABLE, dep_buf); dep_total += len(dep_buf); dep_buf = []
        time.sleep(1)
except KeyboardInterrupt:
    print('Interrupted')
finally:
    if card_buf: ingest_batch(CARD_TABLE, card_buf);    card_total += len(card_buf)
    if dep_buf:  ingest_batch(DEPOSIT_TABLE, dep_buf);  dep_total  += len(dep_buf)
    print(f'\n✅ Done. Card events: {card_total:,} | Deposit events: {dep_total:,}')

## 8. Verify ingestion (KQL)

In [ ]:
verify_kql = f"""
{CARD_TABLE} | summarize events=count(), latest=max(event_time)
| union ({DEPOSIT_TABLE} | summarize events=count(), latest=max(event_time))
"""
result = kusto_client.execute(KUSTO_DATABASE, verify_kql)
for row in result.primary_results[0]:
    print(dict(row.to_dict()))

---

## Optional — Spark Structured Streaming variant

Use this when you need higher throughput (thousands of events/sec). It generates events with `spark-rate` source, maps them to the schema, and writes to Kusto via the Fabric Spark Kusto connector.

In [ ]:
# Spark variant — only run inside a Fabric Spark notebook session
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import json, random, uuid
from datetime import datetime, timezone

# --- mini UDF that returns a card event JSON string ---
def _card_event_json(_):
    e = make_card_event()
    return json.dumps(e, ensure_ascii=False)
card_event_udf = F.udf(_card_event_json, StringType())

rate = (spark.readStream.format('rate')
        .option('rowsPerSecond', 200)  # tune
        .load())

events = rate.withColumn('json', card_event_udf(F.col('value')))

# Parse JSON back into structured columns matching CardTransactions schema
card_schema = ("event_id string, event_time timestamp, card_id string, customer_id string, "
               "card_type string, channel string, merchant_name string, mcc_code string, "
               "mcc_category string, amount_thb double, currency string, auth_status string, "
               "country string, is_fraud_suspect boolean")
card_df = events.select(F.from_json('json', card_schema).alias('e')).select('e.*')

(card_df.writeStream
    .format('com.microsoft.kusto.spark.synapse.datasource')
    .option('kustoCluster', KUSTO_CLUSTER_URI)
    .option('kustoDatabase', KUSTO_DATABASE)
    .option('kustoTable', CARD_TABLE)
    .option('tableCreateOptions', 'CreateIfNotExist')
    .option('checkpointLocation', '/lakehouse/default/Files/_chk/cardtxn')
    .outputMode('append')
    .start())